# Semptom-İlaç İlişkisi Bulgusu

Bilinen psikiyatrik semptom kategorileri ile bu kategorilere karşılık gelen
ziyaretlerde en sık kullanılan ilaçlar arasındaki ilişkiyi, veri setinin
tamamı üzerinden sistematik olarak analiz eder.

**Yöntem:** Her kategori, örnek Türkçe ifadelerle (tohum) temsil edilir. Her
tohum ayrı ayrı `find_similar_phrases` ile tüm semptom kayıtlarına karşı
taranır (ortalama vektör KULLANILMAZ — bkz. tasarım dokümanı, ortalama
almanın gerçek verilerde eşleşmeleri sulandırdığı doğrulandı). Bir kayıt
herhangi bir tohuma eşik üstü benzerse, o kaydın ziyareti kategoriye dahil
edilir.

**Sınırlama:** Bu tanımlayıcı/istatistiksel bir bulgudur — "bu semptom
grubunda bu ilaçlar sık kullanılmış" der, nedensellik ("bu ilaç bu semptomu
tedavi eder") iddia etmez.

## Veri Yükleme

In [ ]:
import sqlite3
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer

sys.path.insert(0, "..")  # trsm/ paketini bulabilmek için (repo kökü)
from trsm.recommendation import compute_drug_statistics
from trsm.symptom_similarity import find_similar_phrases, load_symptom_index

DB_PATH = "../trsm.db"
INDEX_PATH = "../data/symptom_index.npz"
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
MIN_SIMILARITY = 0.75
GUVEN_ESIGI = 90
KUCUK_ORNEKLEM_ESIGI = 20

conn = sqlite3.connect(DB_PATH)
model = SentenceTransformer(MODEL_NAME)
embeddings, ziyaret_ids, phrases = load_symptom_index(INDEX_PATH)

# find_similar_phrases sıralar (argsort) VE top_n uygulanmadan ÖNCE eşik filtresi
# uygulanmaz -- top_n, eşik üstü eşleşme sayısından küçük olursa sonuçları sessizce
# keser (truncation). Bunu önlemek için top_n'i indeksteki toplam kayıt sayısına
# eşitliyoruz, böylece fiili kesme sadece eşik (MIN_SIMILARITY) tarafından yapılır.
TOP_N_PER_SEED = embeddings.shape[0]

print(f"Semptom indeksi: {embeddings.shape[0]} kayıt, {embeddings.shape[1]} boyut")
print(f"TOP_N_PER_SEED = {TOP_N_PER_SEED} (indeksteki toplam kayıt sayısına eşitlendi, kesme riski yok)")


## Semptom Kategorileri

In [ ]:
KATEGORILER = {
    "Uyku bozukluğu": ["uykusuzluk", "uyku düzensizliği", "aşırı uyku hali"],
    "Öfke / ajitasyon": ["öfke nöbeti", "ajitasyon", "saldırganlık", "huzursuzluk"],
    "Anksiyete / gerginlik": ["anksiyete", "gerginlik", "kaygı", "endişe hali"],
    "Depresif belirtiler": ["çökkünlük", "isteksizlik", "mutsuzluk", "enerji azlığı", "ağlama", "üzüntü"],
    "Manik / taşkın belirtiler": ["taşkınlık", "aşırı enerji", "hızlı konuşma", "elevasyon"],
    "Halüsinasyon / hezeyan": ["halüsinasyon", "sesler duyma", "hezeyan", "şüphecilik"],
    "İştah / kilo değişimi": ["iştahsızlık", "aşırı iştah", "kilo alımı", "kilo kaybı"],
    "Sosyal içe kapanma": ["içe kapanma", "sosyal izolasyon", "işlevsellik kaybı"],
    "Madde kullanımı": ["madde kullanımı", "alkol kullanımı", "bağımlılık"],
    "Tedaviye uyumsuzluk": ["ilaç bırakma", "düzensiz ilaç kullanımı", "tedaviyi reddetme"],
}

for kategori, tohumlar in KATEGORILER.items():
    print(f"{kategori}: {len(tohumlar)} tohum ifade")

## Kategori Bazlı Ziyaret Eşleştirme

Her kategori için tohum ifadeler ayrı ayrı sorgulanır ve eşleşen
`ziyaret_id`'lerin birleşimi (union) alınır. Bu, tohum embedding'lerini
ortalamaktan kaçınmak için bilinçli bir tasarım tercihidir.

In [ ]:
def kategori_esleyen_ziyaretler(tohumlar, embeddings, ziyaret_ids, phrases, model, min_similarity, top_n):
    """Her tohumu ayrı sorgu olarak çalıştırır, eşleşen ziyaret_id'lerin
    birleşimini döner. Tohum embedding'lerini ortalamak yerine ayrı ayrı
    sorgulamak bilinçli bir tercih: ortalama, konuşma diline yakın
    ifadelerin (örn. "mutsuzum ağlayasım geliyor") gerçek eşleşmesini
    sulandırıyor (bkz. tasarım dokümanı)."""
    eslenen = set()
    for tohum in tohumlar:
        qemb = model.encode(tohum)
        matches = find_similar_phrases(
            qemb, embeddings, ziyaret_ids, phrases,
            top_n=top_n, min_similarity=min_similarity,
        )
        # Sağlık kontrolü: find_similar_phrases önce top_n ile sınırlayıp SONRA
        # eşik filtresi uyguluyor. Eğer ham sonuç sayısı top_n'e tam eşitse, bu
        # top_n'in gerçek eşik-üstü eşleşme sayısını kestiğinin işareti olabilir.
        if len(matches) == top_n:
            print(
                f"  [UYARI] '{tohum}' tohumu için sonuç sayısı top_n'e ({top_n}) "
                "tam eşit -- olası kesme (truncation), top_n artırılmalı."
            )
        eslenen.update(m["ziyaret_id"] for m in matches)
    return eslenen


kategori_ziyaretleri = {}
for kategori, tohumlar in KATEGORILER.items():
    kategori_ziyaretleri[kategori] = kategori_esleyen_ziyaretler(
        tohumlar, embeddings, ziyaret_ids, phrases, model,
        MIN_SIMILARITY, TOP_N_PER_SEED,
    )

kategori_ozet_df = pd.DataFrame([
    {
        "kategori": kategori,
        "eslesen_ziyaret_sayisi": len(ziyaret_id_seti),
        "kucuk_orneklem": len(ziyaret_id_seti) < KUCUK_ORNEKLEM_ESIGI,
    }
    for kategori, ziyaret_id_seti in kategori_ziyaretleri.items()
]).sort_values("eslesen_ziyaret_sayisi", ascending=False)

kategori_ozet_df


## Kategori Başına İlaç İstatistikleri

Her kategori için eşleşen ziyaretlerdeki en sık kullanılan ilaçlar,
guven eşiğini geçen kayıtlar üzerinden hesaplanır.

In [ ]:
kategori_ilac_satirlari = []
for kategori, ziyaret_id_seti in kategori_ziyaretleri.items():
    istatistikler = compute_drug_statistics(conn, list(ziyaret_id_seti), guven_esigi=GUVEN_ESIGI)
    for satir in istatistikler[:10]:
        kategori_ilac_satirlari.append({
            "kategori": kategori,
            "ilac": satir["ilac"],
            "sayi": satir["sayi"],
            "yuzde": satir["yuzde"],
        })

kategori_ilac_df = pd.DataFrame(kategori_ilac_satirlari)
kategori_ilac_df

In [ ]:
for kategori in KATEGORILER:
    alt_df = kategori_ilac_df[kategori_ilac_df["kategori"] == kategori]
    ziyaret_sayisi = len(kategori_ziyaretleri[kategori])
    uyari = " (KÜÇÜK ÖRNEKLEM)" if ziyaret_sayisi < KUCUK_ORNEKLEM_ESIGI else ""
    print(f"\n{kategori} — {ziyaret_sayisi} ziyaret{uyari}")
    if alt_df.empty:
        print("  (güven eşiğini geçen ilaç kaydı yok)")
    else:
        for _, satir in alt_df.iterrows():
            print(f"  {satir['ilac']}: %{satir['yuzde']} ({satir['sayi']} kayıt)")

## Görselleştirme — Kategori × İlaç Isı Haritası

Kategori bazında ilaç kullanım yüzdelerini ısı haritası ile görselleştir.

**Not:** Kategoriler arası en sık ilaçlar büyük ölçüde örtüşüyor çünkü
kategori-içi yüzdeler, genel reçeteleme sıklığının (hangi ilaçların genel
olarak en çok yazıldığının) izini büyük ölçüde taşıyor. Bunu görünür kılmak
için ısı haritasına, TÜM ziyaretler üzerinden hesaplanan bir "genel" temel
(baseline) satırı eklendi — bir kategorinin bu temel satırdan ne kadar
saptığına bakarak semptoma özgü olabilecek ilişkiler değerlendirilebilir.


In [ ]:
# Genel (tüm ziyaretler) ilaç dağılımı -- kategori yüzdelerini karşılaştırmak
# için bir referans/temel (baseline) satırı olarak kullanılır.
cur = conn.cursor()
cur.execute("SELECT id FROM ziyaret")
tum_ziyaret_ids = [row[0] for row in cur.fetchall()]

genel_istatistikler = compute_drug_statistics(conn, tum_ziyaret_ids, guven_esigi=GUVEN_ESIGI)
GENEL_SATIR_ADI = "Tüm ziyaretler (genel)"
genel_satirlari = [
    {
        "kategori": GENEL_SATIR_ADI,
        "ilac": satir["ilac"],
        "sayi": satir["sayi"],
        "yuzde": satir["yuzde"],
    }
    for satir in genel_istatistikler[:10]
]

kategori_ilac_df_baseline = pd.concat(
    [kategori_ilac_df, pd.DataFrame(genel_satirlari)], ignore_index=True,
)

pivot_df = kategori_ilac_df_baseline.pivot_table(
    index="kategori", columns="ilac", values="yuzde", fill_value=0,
)

# En sık geçen ilk 15 ilaca göre sütunları sınırla (okunabilirlik için)
en_sik_ilaclar = (
    kategori_ilac_df_baseline.groupby("ilac")["sayi"].sum().sort_values(ascending=False).head(15).index
)
pivot_df = pivot_df[[ilac for ilac in en_sik_ilaclar if ilac in pivot_df.columns]]

# Genel satırını en alta sabitle (kategorilerle karışmasın, referans olarak ayrışsın)
kategori_sirasi = [k for k in pivot_df.index if k != GENEL_SATIR_ADI] + [GENEL_SATIR_ADI]
pivot_df = pivot_df.loc[kategori_sirasi]

plt.figure(figsize=(14, 6))
sns.heatmap(pivot_df, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "% (kategori içi)"})
plt.title("Semptom Kategorisi × İlaç Kullanım Yüzdesi")
plt.xlabel("İlaç")
plt.ylabel("Semptom Kategorisi")
plt.tight_layout()
plt.show()


In [ ]:
toplam_eslesen_ziyaret = len(set().union(*kategori_ziyaretleri.values()))
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM ziyaret")
toplam_ziyaret = cur.fetchone()[0]

en_buyuk = kategori_ozet_df.iloc[0]
en_kucuk = kategori_ozet_df.iloc[-1]

kategori_sayilari_toplami = sum(len(s) for s in kategori_ziyaretleri.values())

print("=== Genel Özet ===")
print(f"Toplam ziyaret: {toplam_ziyaret}")
print(f"En az bir kategoriye giren ziyaret: {toplam_eslesen_ziyaret} (%{100*toplam_eslesen_ziyaret/toplam_ziyaret:.1f})")
print(f"En büyük örneklem: {en_buyuk['kategori']} ({en_buyuk['eslesen_ziyaret_sayisi']} ziyaret)")
print(f"En küçük örneklem: {en_kucuk['kategori']} ({en_kucuk['eslesen_ziyaret_sayisi']} ziyaret)")
print(
    f"Kategori sayılarının toplamı: {kategori_sayilari_toplami} "
    "(kategoriler örtüşebilir, bir ziyaret birden fazla kategoriye girebilir)"
)


## Metodolojik Sınırlama

Bu analiz **tanımlayıcı/istatistikseldir**: "bu semptom kategorisiyle
ilişkili ziyaretlerde şu ilaçlar sık kullanılmış" der. Bir ilacın bir
semptomu **tedavi ettiği** yönünde nedensel bir iddia içermez — ilaç
kullanımı ile semptom birlikte gözlenmiştir, ama hangisinin önce geldiği,
aralarındaki nedensellik yönü bu analizden çıkarılamaz. Ayrıca semptom
kategorileri literatürden alınan geniş tema başlıklarıdır, klinik tanı
kategorileri değildir ve tam/eksiksiz olduğu iddia edilmez.

Kategoriler **birbirini dışlamaz**: bir ziyaret birden fazla kategoriye
aynı anda girebilir (örn. hem "Anksiyete / gerginlik" hem "Uyku bozukluğu").
Bu bilinçli bir tasarım tercihidir (bkz. yukarıdaki kategori bazlı eşleştirme
bölümü), ancak bunun sonucu olarak kategori yüzdeleri/sayıları bir bütünün
parçaları olarak okunmamalıdır — toplamları toplam ziyaret sayısını aşabilir.
